# Kaggle Sentiment Training

Fine-tunes `google/muril-base-cased` for Nepali lyric sentiment and writes the scores used by the recommender pipeline.

Run this on Kaggle with GPU and internet enabled.

## 1. Install / import dependencies

In [ ]:
import sys, subprocess

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'datasets', 'accelerate'],
    check=False,
)

import json
import numpy as np
import pandas as pd
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = torch.cuda.is_available()
print('Torch:', torch.__version__)
print('Device:', DEVICE, '| fp16:', USE_FP16)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Config

These mirror `music_rec/config.py` so the artifacts are drop-in compatible.

In [ ]:
BASE_MODEL = 'google/muril-base-cased'
MAX_LEN = 256
EPOCHS = 2
TRAIN_BATCH = 16
EVAL_BATCH = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
INFER_BATCH = 32

## 3. Data-loading helper (checks `/kaggle/input` first, then local)

In [ ]:
from pathlib import Path


def resolve_data_file(*candidates):
    """Return the first existing match from the provided candidates."""
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path

    kaggle_root = Path('/kaggle/input')
    if kaggle_root.exists():
        for candidate in candidates:
            matches = sorted(kaggle_root.rglob(Path(candidate).name))
            if matches:
                return matches[0]

    raise FileNotFoundError(f'Could not locate any of: {candidates}')


WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
print('Working dir for artifacts:', WORK_DIR.resolve())

## 4. Load NepaliSentiment and detect the text and label columns

In [ ]:
from datasets import load_dataset


def load_nepali_sentiment():
    candidates = ['Shushant/NepaliSentiment', 'Shushant/nepali_sentiment']
    last_err = None
    for name in candidates:
        try:
            return load_dataset(name)
        except Exception as e:
            last_err = e
    raise RuntimeError(f'Could not load NepaliSentiment dataset: {last_err}')


ds = load_nepali_sentiment()
split = ds['train'] if 'train' in ds else ds[list(ds.keys())[0]]
cols = split.column_names
text_col = next((c for c in ('text', 'sentence', 'Sentences', 'data') if c in cols), cols[0])
label_col = next((c for c in ('label', 'labels', 'Sentiment', 'sentiment') if c in cols), cols[-1])

raw_labels = sorted({str(x) for x in split[label_col]})
label2id = {lab: i for i, lab in enumerate(raw_labels)}
id2label = {i: lab for lab, i in label2id.items()}
num_labels = len(label2id)

print('Splits:', list(ds.keys()))
print('Detected text column :', text_col)
print('Detected label column:', label_col)
print('label2id:', label2id)

## 5. Tokenize (head truncation is fine for the short training sentences)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def preprocess(batch):
    enc = tokenizer(
        [str(t) for t in batch[text_col]],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    enc['labels'] = [label2id[str(l)] for l in batch[label_col]]
    return enc


tokenized = {k: v.map(preprocess, batched=True, remove_columns=v.column_names) for k, v in ds.items()}
train_split = tokenized['train'] if 'train' in tokenized else list(tokenized.values())[0]
eval_split = tokenized.get('validation') or tokenized.get('test')
print('Train examples:', len(train_split))
print('Eval examples :', len(eval_split) if eval_split is not None else 0)

## 6. Fine-tune with HuggingFace `Trainer` (GPU + fp16)

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

MODEL_DIR = WORK_DIR / 'sentiment_model'

args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    save_strategy='no',
    fp16=USE_FP16,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_split,
    eval_dataset=eval_split,
    data_collator=DataCollatorWithPadding(tokenizer),
)
trainer.train()
if eval_split is not None:
    print('Eval metrics:', trainer.evaluate())

## 7. Save the fine-tuned model dir (+ `label_map.json`)

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
(MODEL_DIR / 'label_map.json').write_text(
    json.dumps(label2id, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Saved fine-tuned model ->', MODEL_DIR.resolve())

## 8. Batch inference over the cleaned lyrics — TAIL truncation

**Critical rule:** lyrics are truncated to the **LAST** `MAX_LEN` tokens.
Song conclusions carry the most emotional weight, so we keep the *end* of
each lyric (drop the beginning) rather than the usual head truncation.

The continuous score mirrors `music_rec/sentiment.py`:

```
sentiment_score = P(positive) - P(negative)   # range [-1, 1]
```

Polarity of each class is inferred from its label name.

In [ ]:
lyrics_path = resolve_data_file(
    'cleaned_lyrics.csv',
    '../music_rec_artifacts/cleaned_lyrics.csv',
    'music_rec_artifacts/cleaned_lyrics.csv',
    'Lyrics_Dataset_final.csv',
)
print('Using lyrics file:', lyrics_path)
df = pd.read_csv(lyrics_path, encoding='utf-8')

if 'lyrics' in df.columns:
    text_series = df['lyrics']
elif 'lyrics_devanagari' in df.columns:
    text_series = df['lyrics_devanagari']
else:
    raise KeyError("No lyrics column found (expected 'lyrics' or 'lyrics_devanagari').")

if 'song_id' in df.columns:
    song_ids = df['song_id'].tolist()
else:
    song_ids = list(range(len(df)))

texts = text_series.fillna('').astype(str).tolist()
print('Songs to score:', len(texts))

In [ ]:
model.eval().to(DEVICE)


def label_polarity(name: str) -> int:
    n = name.lower()
    if any(t in n for t in ('pos', '1', 'happy')):
        return 1
    if any(t in n for t in ('neg', '0', 'sad')):
        return -1
    return 0


pad_id = tokenizer.pad_token_id or 0
scores, labels = [], []

with torch.no_grad():
    for start in range(0, len(texts), INFER_BATCH):
        batch = texts[start:start + INFER_BATCH]
        enc_ids = []
        for t in batch:
            ids = tokenizer.encode(t, add_special_tokens=True)
            if len(ids) > MAX_LEN:
                ids = [ids[0]] + ids[-(MAX_LEN - 1):]
            enc_ids.append(ids)
        maxb = max(len(x) for x in enc_ids)
        input_ids = torch.tensor([x + [pad_id] * (maxb - len(x)) for x in enc_ids]).to(DEVICE)
        attn = torch.tensor([[1] * len(x) + [0] * (maxb - len(x)) for x in enc_ids]).to(DEVICE)
        logits = model(input_ids=input_ids, attention_mask=attn).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        for row in probs:
            pos = sum(row[i] for i in range(len(row)) if label_polarity(id2label[i]) == 1)
            neg = sum(row[i] for i in range(len(row)) if label_polarity(id2label[i]) == -1)
            scores.append(float(pos - neg))
            labels.append(id2label[int(row.argmax())])
        done = min(start + INFER_BATCH, len(texts))
        if done % (INFER_BATCH * 20) < INFER_BATCH or done == len(texts):
            print(f'[sentiment-infer] {done}/{len(texts)}')

## 9. Save `sentiment_scores.csv` + preview

In [ ]:
out = pd.DataFrame({
    'song_id': song_ids,
    'sentiment_label': labels,
    'sentiment_score': scores,
})
scores_path = WORK_DIR / 'sentiment_scores.csv'
out.to_csv(scores_path, index=False, encoding='utf-8')

print('Saved ->', scores_path.resolve())
print('Rows  :', len(out))
print('Score range: [%.3f, %.3f]' % (out['sentiment_score'].min(), out['sentiment_score'].max()))
print('\nArtifacts to download from /kaggle/working:')
print('  - sentiment_scores.csv  -> music_rec_artifacts/sentiment_scores.csv')
print('  - sentiment_model/      -> music_rec_artifacts/sentiment_model/')
out.head(10)